In [100]:
# Imports and base setup
import os
import re
import json
import hashlib
from typing import Dict, List, Tuple

# Must be set BEFORE importing huggingface_hub/transformers
os.environ["HF_HUB_DISABLE_XET"] = "1"

# Patch SSL verification BEFORE importing transformers/huggingface
import urllib3
urllib3.disable_warnings()

# Patch HTTPX to disable SSL verification
try:
    import httpx
    httpx._verify_disabled = True
except ImportError:
    pass

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from sentence_transformers import CrossEncoder
from huggingface_hub import login

In [101]:
import faiss
import numpy as np
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

from pathlib import Path
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv


env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"
load_dotenv(env_path)  # Load environment variables from workspace root .env file


True

In [102]:
# Configure model names and initialize the LLM
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
llm_model_name = "qwen/qwen3-32b"

hf_token = os.getenv("HF_TOKEN", "").strip().strip('"').strip("'")
if hf_token:
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token

llm = ChatGroq(
    model=llm_model_name,
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
 )

print(f"Embedding model: {embedding_model_name}")
print(f"LLM model: {llm_model_name}")

Embedding model: sentence-transformers/all-MiniLM-L6-v2
LLM model: qwen/qwen3-32b


In [103]:
# Load source text
candidates = [
    Path.cwd() / "data.txt",
    Path.cwd().parent / "data.txt",
]

data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("data.txt not found in current directory or parent directory.")

text_data = data_path.read_text(encoding="utf-8")
print(f"Loaded {len(text_data)} characters from {data_path}")

Loaded 50293 characters from c:\projects\learn-rag\vectorDB\data.txt


In [104]:
# Split text into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_text(text_data)
print(f"Created {len(chunks)} chunks")
print(chunks[0][:250])

Created 153 chunks
The Indian Premier League (IPL) is a professional Twenty20 (T20) cricket league in India, organised by the Board of Control for Cricket in India (BCCI).[1] Founded in 2007, it features ten city-based franchise teams.[2] The IPL is the most popular an


In [105]:
chunks

['The Indian Premier League (IPL) is a professional Twenty20 (T20) cricket league in India, organised by the Board of Control for Cricket in India (BCCI).[1] Founded in 2007, it features ten city-based franchise teams.[2] The IPL is the most popular and richest cricket league in the world and the 11th richest sporting league in the world by revenue. It is held annually between March and May',
 '. It is held annually between March and May. It has an exclusive window in the Future Tours Programme of the International Cricket Council, resulting in fewer international tours occurring during the seasons.[3] It is also the most viewed Indian sports event, per the Broadcast Audience Research Council.[4][5]',
 'In 2010, the IPL became the first sporting event to broadcast live on YouTube.[6][7] In 2014, it ranked sixth in attendance among all sports leagues.[8] Inspired by the success of the IPL, other Indian sports leagues have been established.[a][11][12] The IPL is the second-richest sports

In [106]:
from langchain_text_splitters import SpacyTextSplitter

spacy_splitter = SpacyTextSplitter(
    pipeline="sentencizer",
    chunk_size=500,
    chunk_overlap=80,
)
spacy_chunks = spacy_splitter.split_text(text_data)

Created a chunk of size 877, which is longer than the specified 500
Created a chunk of size 917, which is longer than the specified 500
Created a chunk of size 569, which is longer than the specified 500
Created a chunk of size 535, which is longer than the specified 500
Created a chunk of size 2017, which is longer than the specified 500
Created a chunk of size 1630, which is longer than the specified 500
Created a chunk of size 1068, which is longer than the specified 500
Created a chunk of size 794, which is longer than the specified 500
Created a chunk of size 2105, which is longer than the specified 500
Created a chunk of size 4679, which is longer than the specified 500
Created a chunk of size 1771, which is longer than the specified 500
Created a chunk of size 2464, which is longer than the specified 500
Created a chunk of size 1725, which is longer than the specified 500
Created a chunk of size 1685, which is longer than the specified 500
Created a chunk of size 1127, which is 

In [107]:
spacy_chunks

['The Indian Premier League (IPL) is a professional Twenty20 (T20) cricket league in India, organised by the Board of Control for Cricket in India (BCCI).[1] Founded in 2007, it features ten city-based franchise teams.[2] The IPL is the most popular and richest cricket league in the world and the 11th richest sporting league in the world by revenue.\n\nIt is held annually between March and May.',
 'It has an exclusive window in the Future Tours Programme of the International Cricket Council, resulting in fewer international tours occurring during the seasons.[3] It is also the most viewed Indian sports event, per the Broadcast Audience Research Council.[4][5]\n\nIn 2010, the IPL became the first sporting event to broadcast live on YouTube.[6][7] In 2014, it ranked sixth in attendance among all sports leagues.[8] Inspired by the success of the IPL, other Indian sports leagues have been established.[a][11][12] The IPL is the second-richest sports league in the world by per-match value, a

In [108]:
# Load a very lightweight Hugging Face embedding model
embedder = SentenceTransformer(embedding_model_name, device="cpu")

print(f"Loaded embedder: {embedding_model_name}")
print(f"Embedding dimension: {embedder.get_embedding_dimension()}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2816.09it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded embedder: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension: 384


In [109]:
from langchain_core.embeddings import Embeddings

class SentenceTransformerEmbeddings(Embeddings):
    def __init__(self, model):
        self.model = model

    def embed_documents(self, texts):
        return self.model.encode(texts, convert_to_numpy=True).tolist()

    def embed_query(self, text):
        return self.model.encode([text], convert_to_numpy=True)[0].tolist()

embedding_adapter = SentenceTransformerEmbeddings(embedder)


In [110]:
vectors = embedding_adapter.embed_documents(chunks)
print(f"Chunk vectors shape: {len(vectors)} x {len(vectors[0])}")

Chunk vectors shape: 153 x 384


In [111]:
vectors

[[-0.01985953375697136,
  -0.016603684052824974,
  -0.00924310740083456,
  -0.01212960947304964,
  0.007799839600920677,
  0.019257327541708946,
  0.01656925491988659,
  0.040482353419065475,
  0.12066049128770828,
  0.05855243280529976,
  -0.03957574442028999,
  -0.0008269763784483075,
  0.031179487705230713,
  0.05924023315310478,
  0.0596400611102581,
  -0.0380207821726799,
  -0.028116777539253235,
  -0.06889108568429947,
  -0.025218434631824493,
  -0.14184074103832245,
  0.02357042208313942,
  -0.0034080510959029198,
  -0.037415117025375366,
  -0.0034558684565126896,
  -0.012312146835029125,
  -0.0255296491086483,
  -0.025712816044688225,
  0.060062479227781296,
  -0.03267532214522362,
  -0.03931235894560814,
  0.023988151922822,
  0.06447097659111023,
  0.03499160706996918,
  0.07658300548791885,
  -0.10433320701122284,
  -0.0808817595243454,
  -0.04005021974444389,
  0.03784435987472534,
  0.04247209057211876,
  -0.018582027405500412,
  0.04120081290602684,
  -0.0951550304889679,

In [112]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

embedding_dim = embedder.get_embedding_dimension()
index = faiss.IndexFlatL2(embedding_dim)

vector_store = FAISS(
    embedding_function=embedding_adapter,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

print(f"Vector store initialized with dimension: {embedding_dim}")

Vector store initialized with dimension: 384


There are 2 approaches like we can precompute our embeddings and send to the VectorDB or we can directly send our details to the vectorDB it will use Sentence transformer and automatically embedd those documents. 

In [114]:
from uuid_utils import uuid4

uuids = [str(uuid4()) for _ in range(len(vectors))]
text_embeddings = list(zip(chunks, vectors))

vector_store.add_embeddings(text_embeddings=text_embeddings, ids=uuids)
print(f"Added {len(uuids)} embeddings to FAISS")

Added 153 embeddings to FAISS


In [ ]:
# Save FAISS vector store to disk and reload for inference
store_dir = Path.cwd() / "faiss_store"
store_dir.mkdir(parents=True, exist_ok=True)

vector_store.save_local(folder_path=str(store_dir))
print(f"Saved vector store to: {store_dir}")

loaded_vector_store = FAISS.load_local(
    folder_path=str(store_dir),
    embeddings=embedding_adapter,
    allow_dangerous_deserialization=True,
 )

query = "when did SRH win their first IPL title ?"
results = loaded_vector_store.similarity_search(query, k=5)

print("Top 3 retrieved chunks from loaded store:")
for i, doc in enumerate(results, start=1):
    print(f"\nResult {i}:\n{doc.page_content[:1000]}")

Saved vector store to: c:\projects\learn-rag\vectorDB\faiss_store
Top 3 retrieved chunks from loaded store:

Result 1:
players.[60] Royal Challengers Bengaluru won their first IPL title in 2025, receiving ₹20 crore, while runners-up Punjab Kings earned ₹12 crore.[61]

Result 2:
A match during the 2008 IPL inaugural season in Chennai
Expansions and terminations

Result 3:
. In 2022 IPL, the league expanded again with the introduction of Gujarat Titans and Lucknow Super Giants, making it a ten-team tournament. Over time, some teams underwent rebranding, such as Delhi Daredevils becoming Delhi Capitals in 2019 and Kings XI Punjab rebranding as Punjab Kings in 2021. Chennai Super Kings and Mumbai Indians remain the most successful franchises, winning five IPL titles each.[71] As of the 2025 season, the league consists of 10 teams.

Result 4:
Several IPL franchise owners have expanded their business by acquiring teams in other franchise leagues, such as the West Indies' Caribbean Premier Le

In [122]:
from pathlib import Path
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv


env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"
load_dotenv(env_path)  # Load environment variables from workspace root .env file


llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
)

In [124]:
question = HumanMessage('who won the IPL title in 2016 ?')
system = SystemMessage(f'You are a knowledgeable assistant. You answer in short sentences answer only from the given context dont answer anything out of the context {results}')

messages = [system, question]
response = llm.invoke(messages)

print(response.content)

The provided documents do not mention the 2016 IPL title winner. The information available focuses on recent seasons (e.g., Royal Challengers Bengaluru winning in 2025) and historical expansions/rebranding, but 2016 is not referenced.
